In [ ]:
#----- Setup -----#

# 3rd party imports
import copy
import os
import torch
import torch.nn as nn
import torch.optim as optim
import wandb

from dotenv import load_dotenv
from torch.utils.data import DataLoader, random_split, TensorDataset

# Local imports
from transportation_models.utils.logs import make_logger

# WandB setup
wandb_entity="transportation-models"
wandb_project_name="PeMS-Imputation"
load_dotenv()
wandb.login(key=os.getenv("WANDB_API_KEY"))

# Logging setup
logger = make_logger(include_stdout=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/robstallman/.netrc
wandb: Currently logged in as: robstallman3 (rost5691-cu-boulder) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
#----- Data loading -----#

# Load dataset
data_filepath = "/Volumes/easystore/work/boulder/Caltrans/PeMS/processed_data/hour_lookback_15min_horizon.pt"
data = torch.load(data_filepath)
X,y = data['X'], data['y']
dataset = TensorDataset(X.float(),y.float())
logger.info("Loaded dataset with length: ", len(dataset))

# Create training, validation, and testing splits
splits = [0.6, 0.2, 0.2]
train_set, val_set, test_set = random_split(dataset, lengths=splits)

/var/folders/7m/3r23lm6s517bl9502bgqp64w0000gn/T/ipykernel_24345/1396610036.py:9: UserWarning: Length of split at index 1 is 0. This might result in an empty dataset.
  train_set, val_set, test_set = random_split(dataset, lengths=splits)
/var/folders/7m/3r23lm6s517bl9502bgqp64w0000gn/T/ipykernel_24345/1396610036.py:9: UserWarning: Length of split at index 2 is 0. This might result in an empty dataset.
  train_set, val_set, test_set = random_split(dataset, lengths=splits)


In [23]:
dataset = TensorDataset(X.float(),y.float())
logger.info("Loaded dataset with length: ", len(dataset))

# Create training, validation, and testing splits
splits = [0.6, 0.2, 0.2]
train_set, val_set, test_set = random_split(dataset, lengths=splits)

: 

In [ ]:
# Basic GRU model
class simpleGRU(nn.Module):
    def __init__(self, input_size=16, hidden_size=64, output_steps=3, output_size=2):
        super(simpleGRU, self).__init__()
        self.hidden_size = hidden_size
        self.output_steps = output_steps
        self.output_size = output_size

        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_steps * output_size)

    def forward(self, x, hidden=None):
        # x: [batch, 12, 16]
        batch_size = x.size(0)

        if hidden is None:
            hidden = torch.zeros(1, batch_size, self.hidden_size, device=x.device)

        gru_out, hidden = self.gru(x, hidden)       # [batch, 12, hidden]
        last_step = gru_out[:, -1, :]               # [batch, hidden]

        out = self.fc(last_step)                    # [batch, 6]
        out = out.view(batch_size, self.output_steps, self.output_size)  # [batch, 3, 2]

        return out, hidden

# Define a function to save models
def save_model(
    model: nn.Module,
    name: str,
    root_path: str = os.getcwd(),
) -> str:
    # Create a directory for models if it doesn't yet exist
    if not os.path.exists(os.path.join(root_path, "models")):
        os.mkdir(os.path.join(root_path, "models"))

    # Save the model to the directory
    filepath = os.path.join(root_path, "models", f"{name}.pt")
    torch.save(model.state_dict(), filepath)
    logger.info(f"Model saved to: {filepath}")

    # Return the filepath
    return filepath

# Define a function to evaluate a single epoch
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    loss_fn: nn.Module,
    optimizer: optim.Optimizer,
    train: bool = True,
) -> float:
    # Make sure we're on the correct device
    model = model.to(device)

    # Set the model mode – either training or evaluation
    if train:
        model.train()
    else:
        model.eval()

    # Initial values for tracking loss and accuracy over the epoch
    running_loss = 0.0

    # Set context based on model mode
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        # Iterate through the batches in the loader
        for xb, yb in loader:
            # Transfer to device
            xb, yb = xb.to(device), yb.to(device)

            # Zero gradients
            optimizer.zero_grad()

            # -- Forward pass -- #
            # Get model predictions
            output, hidden = model(xb)

            # Evaluate loss
            loss = loss_fn(output, yb)

            # -- Backward pass -- #
            if train:
                # Update gradients
                loss.backward()

                # Adjust learning weights
                optimizer.step()

            # Update running counts for loss and accuracy
            running_loss += loss.item()

    # Calculate average batch loss
    running_loss = running_loss / len(loader)

    # Return loss and accuracy for this epoch
    return running_loss

# Define function for training
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    loss_fn: nn.Module,
    optimizer: optim.Optimizer,
    n_epochs: int = 1500,
    early_stopping: bool = True,
    patience: int = 100,
    min_delta: float = 1e-2,
    wandb_config: dict = {},
    wandb_tags: list[str] = [],
    wandb_notes: str = "",
    model_name: str = "baseline",
) -> None:
    # Early stopping setup
    best_val_loss = float('inf')
    best_model_weights = copy.deepcopy(model.state_dict())
    no_improvement_count = 0

    # W&B setup
    wandb_config["epochs"] = n_epochs
    wandb_config["batch_size"] = train_loader.batch_size
    wandb_config["early_stopping"] = early_stopping
    wandb_config["patience"] = patience
    wandb_config["min_delta"] = min_delta

    # Start training
    with wandb.init(
        entity=wandb_entity,
        project=wandb_project_name,
        notes=wandb_notes,
        tags=wandb_tags,
        config=wandb_config,
    ) as run:
        for epoch in range(n_epochs):
            # Training over epoch
            train_loss = run_epoch(
                model=model,
                loader=train_loader,
                device=device,
                loss_fn=loss_fn,
                optimizer=optimizer,
                train=True,
            )

            # Validation over epoch
            val_loss = run_epoch(
                model=model,
                loader=val_loader,
                device=device,
                loss_fn=loss_fn,
                optimizer=optimizer,
                train=False,
            )

            # Save losses for this epoch
            run.log(
                {
                    "training_loss": train_loss,
                    "validation_loss": val_loss,
                }
            )

            # Early stopping logic
            if early_stopping:
                # Improvement means val_loss got smaller by at least min_delta
                if val_loss < best_val_loss - min_delta:
                    best_val_loss = val_loss
                    best_model_weights = copy.deepcopy(model.state_dict())
                    no_improvement_count = 0
                else:
                    no_improvement_count += 1
                    if no_improvement_count >= patience:
                        break

        # Reload best model
        if early_stopping:
            model.load_state_dict(best_model_weights)

        # Upload the best model as an artifact
        model_filepath = save_model(model=model, name=model_name)
        run.log_artifact(model_filepath, name="trained-model", type="model")

    return

In [19]:
#----- Model inspection -----#
batch_size = 8
x_batch = torch.randn(batch_size, 12, 16)

model = simpleGRU(input_size=16, hidden_size=64, output_steps=3, output_size=2)
output, hidden = model(x_batch)

print("Input shape:\n", x_batch.shape)  # [8, 12, 16]
print("Output shape:\n", output.shape)   # [8, 3, 2]
print("Example output:\n", output[0])

Input shape:
 torch.Size([8, 12, 16])
Output shape:
 torch.Size([8, 3, 2])
Example output:
 tensor([[-0.1230, -0.2025],
        [-0.1597,  0.0386],
        [ 0.0274,  0.1029]], grad_fn=<SelectBackward0>)


In [21]:
#----- Model training -----#

# Define hyperparameters
batch_size = 16
lr = 0.01
n_epochs = 10

# Set up DataLoaders
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

# Define model, loss function, and optimizer
model = simpleGRU()
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=lr)

# Train model
train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    loss_fn=loss_fn,
    optimizer=optimizer,
    n_epochs=n_epochs,
    early_stopping=False,
    wandb_config = {
        "model": "simpleGRU",
        "loss_function": "MSE",
        "optimizer": "SGD",
        "learning_rate":lr,
    },
    wandb_tags=["prototyping"],
    model_name="simpleGRU"
    )

2026-03-03 14:35:08,791 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): api.wandb.ai:443
2026-03-03 14:35:08,897 - urllib3.connectionpool - DEBUG - https://api.wandb.ai:443 "POST /graphql HTTP/1.1" 200 None
2026-03-03 14:35:08,954 - urllib3.connectionpool - DEBUG - https://api.wandb.ai:443 "POST /graphql HTTP/1.1" 200 None
2026-03-03 14:35:09,317 - git.cmd - DEBUG - Popen(['git', 'version'], cwd=/Users/robstallman/projects/transportation-models/src/transportation_models/scripts, stdin=None, shell=False, universal_newlines=False)
2026-03-03 14:35:09,343 - git.cmd - DEBUG - Popen(['git', 'version'], cwd=/Users/robstallman/projects/transportation-models/src/transportation_models/scripts, stdin=None, shell=False, universal_newlines=False)
2026-03-03 14:35:09,361 - git.util - DEBUG - sys.platform='darwin', git_executable='git'
2026-03-03 14:35:09,363 - git.cmd - DEBUG - Popen(['git', 'cat-file', '--batch-check'], cwd=/Users/robstallman/projects/transportation-models, s

Traceback (most recent call last):
  File "/var/folders/7m/3r23lm6s517bl9502bgqp64w0000gn/T/ipykernel_24345/204651872.py", line 140, in train_model
    train_loss = run_epoch(
  File "/var/folders/7m/3r23lm6s517bl9502bgqp64w0000gn/T/ipykernel_24345/204651872.py", line 79, in run_epoch
    predictions = model(xb)
  File "/Users/robstallman/.pyenv/versions/transportation-models-env/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/Users/robstallman/.pyenv/versions/transportation-models-env/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
  File "/var/folders/7m/3r23lm6s517bl9502bgqp64w0000gn/T/ipykernel_24345/204651872.py", line 19, in forward
    gru_out, hidden = self.gru(x, hidden)       # [batch, 12, hidden]
  File "/Users/robstallman/.pyenv/versions/transportation-models-env/lib/python3.10/site-packages/torch/nn/modu

ValueError: input must have the type torch.float32, got type torch.float64